# Factory bronze layer

### Imports

In [0]:
import requests
from pyspark.sql.functions import current_timestamp
from delta.tables import DeltaTable

### Parameters

In [0]:
dbutils.widgets.text("entity_name", "factory")
dbutils.widgets.text("load_pattern", "1")
dbutils.widgets.text("SUBGRAPH_URL", "https://gateway.thegraph.com/api/a5bbc98aac75dac555f9aba8747c7e2e/subgraphs/id/5zvR82QoaXYFyDEKLZ9t6v9adgnptxYpKpSbxtgVENFV")

In [0]:
SUBGRAPH_URL = dbutils.widgets.get("SUBGRAPH_URL")
entity_name = dbutils.widgets.get("entity_name")
load_pattern = dbutils.widgets.get("load_pattern")

### Variables

In [0]:
query_variables = {
    "MAINNET_FACTORY_ID": "0x1F98431c8aD98523631AE4a59f267346ea31F984"
}


#factory

In [0]:
factory_query = """
query FactoryQuery($MAINNET_FACTORY_ID: ID!){
    factory(id: $MAINNET_FACTORY_ID ) {
    id
    poolCount
    txCount
    totalFeesUSD
    totalFeesETH
    totalVolumeUSD
    totalVolumeETH
    totalValueLockedUSD
    totalValueLockedETH
  }
}
"""

In [0]:
response = requests.post(SUBGRAPH_URL, json={"query": factory_query, "variables": query_variables})

In [0]:
factory_df = spark.createDataFrame([response.json()["data"][entity_name]])
factory_df = factory_df.withColumn("load_timestamp", current_timestamp())

In [0]:
display(factory_df)

In [0]:
if load_pattern == "1":
    print("*INFO*: Executing full load.")
    factory_df.write.format("delta")\
        .mode("overwrite")\
        .option("overwriteSchema", "true")\
        .saveAsTable(f"uniswap.bronze.{entity_name}")
    dbutils.notebook.exit(f"Entity {entity_name} loaded correctly with pattern {load_pattern}.")

elif load_pattern == "2":
    print("*INFO*: Executing incremental load.")
    factory_df.write.format("delta")\
        .mode("append")\
        .saveAsTable(f"uniswap.bronze.{entity_name}")
    dbutils.notebook.exit(f"Entity {entity_name} loaded correctly with pattern {load_pattern}.")

elif load_pattern == "3":
    print("*INFO*: Executing differential load.")
    old_factory_table = DeltaTable.forName(spark, f"uniswap.bronze.{entity_name}")
    old_factory_table.alias("old").merge(
        factory_df.alias("new"),
        "old.id = new.id"
    )\
    .whenMatchedUpdateAll()\
    .whenNotMatchedInsertAll()\
    .execute()
    dbutils.notebook.exit(f"Entity {entity_name} loaded correctly with pattern {load_pattern}.")
else:
    print("*ERROR*: Invalid load pattern.")
    dbutils.notebook.exit("Invalid load pattern.")